### Objective

This notebook is the EMA cross sweep harness.

It proves one complete sweeprun:

- create or load a sweep and sweeprun record
- build sweeprun and bot config from the record
- load this sweeprun's data window
- precompute signals once
- run the sweep runtime over snapshots
- save results back to the database
- show raw results and a final chart/report

This notebook is sweep-only. Live/simnet/backtest runners can share components later, but they are not forced into this harness.


In [ ]:
# Sweep Control
# User-edited sweep template plus notebook-only controls.

template = """
[sweep]
mode = "fast"
start_bot_id = 1

[params]
fast = [9]
slow = [21]

[botrun.runtime]
bot_id = 1
mode = "sweep"
max_loop = 0
loop_seconds = 1.0

[botrun.market]
symbol = "BTCUSDT"
interval = "1m"

[botrun.backtest]
start = "2025-01-01"
stop = "2025-12-31T23:59:59"

[[botrun.signalers]]
name = "emacross"
interval = "1m"

[botrun.signalers.params]
fast = 9
slow = 21

[botrun.executor]
name = "tradebot"
take_profit_pct = 2.0
stop_loss_pct = 1.0
max_cycles = 0

[botrun.risk]
score = 1
"""

# 0 creates a new sweep. Nonzero loads and reruns that sweep_id.
SWEEP_ID = 0
FEE_PCT = 0.0


In [ ]:
# Imports
# All notebook imports live here.

from dataclasses import dataclass
from datetime import datetime, timezone
import itertools
import json
from pathlib import Path
import time
import tomllib

from IPython.display import HTML, IFrame, display
from sqlalchemy import select

from nuubot import Nuubot
from nuubot.core.dtypes import Bar, MarketSnapshot, Signal
from nuubot.core.market_data import date_ms, load_binance_bars
from nuubot.core.models.mconfig import BotrunConfig, SweepConfig
from nuubot.datastore import SweepRow, SweeprunRow


In [ ]:
# Prepare SweepRun
# Parse template, validate it, create or load sweep records, then load all sweeprun records.

def scalar_json(config: BotrunConfig) -> str:
    return json.dumps(config.model_dump(mode="json"), sort_keys=True, separators=(",", ":"))


def apply_param(botrun: dict, key: str, value) -> None:
    signaler_params = botrun["signalers"][0].setdefault("params", {})
    if key in signaler_params:
        signaler_params[key] = value
        return
    if key in botrun["executor"]:
        botrun["executor"][key] = value
        return
    raise ValueError(f"sweep param has no target in botrun: {key}")


def expand_botruns(config: SweepConfig) -> list[BotrunConfig]:
    params = config.params or {}
    keys = list(params)
    values = [value if isinstance(value, list) else [value] for value in params.values()]
    expanded = []
    for combo in itertools.product(*values) if values else [()]:
        botrun = config.botrun.model_dump(mode="json")
        for key, value in zip(keys, combo, strict=True):
            apply_param(botrun, key, value)
        expanded.append(BotrunConfig.model_validate(botrun))
    return expanded


def create_sweep_record(nuubot, config: SweepConfig, manifest_json: str) -> int:
    botruns = expand_botruns(config)
    with nuubot.datastore.session(nuubot.config.databases.sweeps) as session:
        sweep = SweepRow(
            sweep_desc="emacross_notebook_smoke",
            config_json=manifest_json,
            results_json="{}",
            status="configured",
            sweeprun_count=len(botruns),
        )
        session.add(sweep)
        session.flush()
        for index, botrun in enumerate(botruns):
            session.add(SweeprunRow(
                sweep_id=sweep.sweep_id,
                sweeprun_index=index,
                config_json=scalar_json(botrun),
                results_json="{}",
                status="configured",
            ))
        session.commit()
        return sweep.sweep_id


def load_sweep_records(nuubot, sweep_id: int) -> tuple[SweepRow, list[SweeprunRow]]:
    with nuubot.datastore.session(nuubot.config.databases.sweeps) as session:
        sweep = session.get(SweepRow, sweep_id)
        if sweep is None:
            raise ValueError(f"sweep not found: {sweep_id}")
        sweepruns = list(session.scalars(select(SweeprunRow).where(SweeprunRow.sweep_id == sweep_id).order_by(SweeprunRow.sweeprun_index)))
        if not sweepruns:
            raise ValueError(f"sweepruns not found for sweep: {sweep_id}")
        session.expunge(sweep)
        for row in sweepruns:
            session.expunge(row)
        return sweep, sweepruns


nuubot = Nuubot().setup()
try:
    if SWEEP_ID == 0:
        template_data = tomllib.loads(template)
        template_data["botrun"]["backtest"]["data_dir"] = f"{nuubot.config.paths.data_dir}/binance/raw/spot/monthly/klines"
        template_config = SweepConfig.model_validate(template_data)
        manifest_json = json.dumps(template_data, sort_keys=True, separators=(",", ":"))
        sweep_id = create_sweep_record(nuubot, template_config, manifest_json)
    else:
        sweep_id = SWEEP_ID

    sweep_record, sweeprun_records = load_sweep_records(nuubot, sweep_id)
finally:
    nuubot.stop()

print(f"ready sweep_id={sweep_record.sweep_id} sweepruns={len(sweeprun_records)}")

In [ ]:
# Configs
# Tiny notebook-local wrappers over each loaded sweeprun record.

class SweeprunConfig:
    def __init__(self, record: SweeprunRow) -> None:
        self.record = record
        self.sweeprun_id = record.sweeprun_id
        self.sweep_id = record.sweep_id
        self.symbol = BotrunConfig.model_validate(json.loads(record.config_json)).market.symbol


class BotConfig:
    def __init__(self, record: SweeprunRow) -> None:
        self.record = record
        self.config = BotrunConfig.model_validate(json.loads(record.config_json))


sweeprun_record = sweeprun_records[0]
sweeprun_config = SweeprunConfig(sweeprun_record)
bot_config = BotConfig(sweeprun_record)

In [5]:
# Data
# Load this sweeprun's bars and yield point-in-time snapshots.

class Data:
    def __init__(self, sweeprun_config: SweeprunConfig, bot_config: BotConfig) -> None:
        self.sweeprun_config = sweeprun_config
        self.config = bot_config.config
        self.interval = self.config.market.interval
        self.bars: list[Bar] = []
        self.load_ms = 0

    def init(self) -> None:
        started = time.perf_counter()
        self.bars = load_binance_bars(self.config)
        self.load_ms = int((time.perf_counter() - started) * 1000)

    def __iter__(self):
        for bar in self.bars:
            yield MarketSnapshot(bars={self.interval: bar})

    def results(self) -> dict:
        return {"load_ms": self.load_ms, "loaded_bars": len(self.bars)}


In [6]:
# Signaler Under Test
# EMA cross signaler. init(data) precomputes indicators and signals once.

class EmaCross:
    def __init__(self, bot_config: BotConfig) -> None:
        config = bot_config.config
        signaler_config = config.signalers[0]
        self.interval = signaler_config.interval
        self.fast = int(signaler_config.params["fast"])
        self.slow = int(signaler_config.params["slow"])
        self.partial = signaler_config.partial
        if self.fast <= 0 or self.slow <= 0:
            raise ValueError("EMA periods must be positive")
        if self.fast >= self.slow:
            raise ValueError("fast EMA must be lower than slow EMA")
        self.rows: dict[int, dict] = {}
        self.load_ms = 0
        self.warmup_bars = max(self.fast, self.slow) + 10

    def init(self, data: Data) -> None:
        start_ms = date_ms(data.config.backtest.start)
        warmup_count = sum(1 for bar in data.bars if bar.ts_ms < start_ms)
        if warmup_count < self.warmup_bars:
            raise RuntimeError(f"not enough warmup bars: need={self.warmup_bars} got={warmup_count}")

        started = time.perf_counter()
        fast_ema = None
        slow_ema = None
        previous_diff = None
        count = 0
        for bar in data.bars:
            signal = Signal()
            if bar.closed or self.partial:
                count += 1
                fast_ema = ema(fast_ema, bar.close, self.fast)
                slow_ema = ema(slow_ema, bar.close, self.slow)
                if count >= self.slow:
                    diff = fast_ema - slow_ema
                    if previous_diff is not None:
                        if previous_diff <= 0 < diff:
                            signal = Signal(entry=True, reason="ema_cross_up")
                        elif previous_diff >= 0 > diff:
                            signal = Signal(exit=True, reason="ema_cross_down")
                    previous_diff = diff
            self.rows[bar.ts_ms] = {"signal": signal, "ema_fast": fast_ema, "ema_slow": slow_ema}
        self.load_ms = int((time.perf_counter() - started) * 1000)
        self.warmup_count = warmup_count

    async def next(self, snapshot: MarketSnapshot) -> Signal:
        bar = snapshot.bars[self.interval]
        return self.rows[bar.ts_ms]["signal"]

    def values(self, ts_ms: int) -> dict:
        return self.rows[ts_ms]

    def results(self) -> dict:
        return {"signal_load_ms": self.load_ms, "warmup_bars": self.warmup_count}


def ema(previous: float | None, price: float, period: int) -> float:
    if previous is None:
        return price
    alpha = 2 / (period + 1)
    return price * alpha + previous * (1 - alpha)


In [7]:
# Risk Under Test
# Placeholder risk profile. Kept as a class because risk rules will grow here.

class Risk:
    def __init__(self, bot_config: BotConfig) -> None:
        self.config = bot_config.config

    async def init(self) -> None:
        pass

    async def next(self, snapshot: MarketSnapshot, signal: Signal, accounts: dict[str, "RuntimeAccount"]) -> Signal:
        return signal

    def results(self) -> dict:
        return {}


In [8]:
# Executor Under Test
# Tradebot strategy. It owns trade state; runtime owns when next() is called.

@dataclass
class RuntimeAccount:
    role: str
    name: str
    recon_count: int = 0

    async def recon(self, snapshot: MarketSnapshot) -> None:
        self.recon_count += 1


@dataclass
class TradeResult:
    pnl_pct: float
    fees_pct: float
    wins: int
    losses: int
    trades: int
    cycles: int


class Tradebot:
    def __init__(self, bot_config: BotConfig) -> None:
        config = bot_config.config.executor
        self.take_profit_pct = config.take_profit_pct
        self.stop_loss_pct = config.stop_loss_pct
        self.max_cycles = config.max_cycles
        self.fee_pct = FEE_PCT
        self.active = False
        self.entry_price = 0.0
        self.pnl_pct = 0.0
        self.fees_pct = 0.0
        self.wins = 0
        self.losses = 0
        self.trades = 0
        self.cycles = 0
        self.events: list[dict] = []
        self.equity: list[tuple[int, float]] = []

    async def init(self) -> dict[str, RuntimeAccount]:
        return {"trade": RuntimeAccount(role="trade", name="trade")}

    async def next(self, snapshot: MarketSnapshot, risk: Signal, accounts: dict[str, RuntimeAccount]) -> None:
        bar = snapshot.bars[next(iter(snapshot.bars))]
        if self.active:
            change_pct = self._change_pct(bar.close)
            if risk.exit:
                self._close(bar, change_pct, risk.reason)
            elif self.take_profit_pct > 0 and change_pct >= self.take_profit_pct:
                self._close(bar, change_pct, "take_profit")
            elif self.stop_loss_pct > 0 and change_pct <= -self.stop_loss_pct:
                self._close(bar, change_pct, "stop_loss")

        if not self.active and risk.entry and self._can_enter():
            self._open(bar, risk.reason)

        open_pnl = self._change_pct(bar.close) if self.active else 0.0
        self.equity.append((bar.ts_ms, self.pnl_pct + open_pnl))

    async def stop(self, bar: Bar | None) -> None:
        if self.active and bar is not None:
            self._close(bar, self._change_pct(bar.close), "stop")

    def result(self) -> TradeResult:
        return TradeResult(self.pnl_pct, self.fees_pct, self.wins, self.losses, self.trades, self.cycles)

    def _open(self, bar: Bar, reason: str) -> None:
        self.active = True
        self.entry_price = bar.close
        self.trades += 1
        self.events.append({"event": "entry", "ts_ms": bar.ts_ms, "price": bar.close, "reason": reason})

    def _close(self, bar: Bar, change_pct: float, reason: str) -> None:
        fee_pct = self.fee_pct * 2
        net_pct = change_pct - fee_pct
        self.pnl_pct += net_pct
        self.fees_pct += fee_pct
        self.wins += int(net_pct >= 0)
        self.losses += int(net_pct < 0)
        self.cycles += 1
        self.active = False
        self.entry_price = 0.0
        self.events.append({"event": "exit", "ts_ms": bar.ts_ms, "price": bar.close, "reason": reason, "pnl_pct": net_pct, "fees_pct": fee_pct})

    def _can_enter(self) -> bool:
        return self.max_cycles == 0 or self.cycles < self.max_cycles

    def _change_pct(self, price: float) -> float:
        return (price - self.entry_price) / self.entry_price * 100


In [9]:
# SweepRuntime
# Orchestrates one sweeprun: init components, loop snapshots, call component next(), collect results.

class SweepRuntime:
    def __init__(self, sweeprun_config: SweeprunConfig, bot_config: BotConfig, data: Data, signaler: EmaCross, risk: Risk, executor: Tradebot) -> None:
        self.sweeprun_config = sweeprun_config
        self.bot_config = bot_config
        self.config = bot_config.config
        self.data = data
        self.signaler = signaler
        self.risk = risk
        self.executor = executor
        self.accounts: dict[str, RuntimeAccount] = {}
        self.start_ms = date_ms(self.config.backtest.start)
        self.stop_ms = date_ms(self.config.backtest.stop)
        self.last_bar = None
        self.chart_rows = []
        self.signal_count = 0
        self.signal_ms = 0
        self.execution_ms = 0
        self.loop_ms = 0
        self.elapsed_ms = 0

    async def init(self) -> None:
        started = time.perf_counter()
        self.data.init()
        self.signaler.init(self.data)
        await self.risk.init()
        self.accounts = await self.executor.init()
        self.elapsed_ms = int((time.perf_counter() - started) * 1000)

    async def run(self) -> None:
        started = time.perf_counter()
        for snapshot in self.data:
            bar = snapshot.bars[self.config.market.interval]
            if bar.ts_ms > self.stop_ms:
                break
            await self.next(snapshot)
        self.loop_ms = int((time.perf_counter() - started) * 1000)
        await self.executor.stop(self.last_bar)

    async def next(self, snapshot: MarketSnapshot) -> None:
        bar = snapshot.bars[self.config.market.interval]
        signal_started = time.perf_counter()
        signal = await self.signaler.next(snapshot)
        self.signal_ms += int((time.perf_counter() - signal_started) * 1000)
        if bar.ts_ms < self.start_ms:
            return

        for account in self.accounts.values():
            await account.recon(snapshot)
        risk_signal = await self.risk.next(snapshot, signal, self.accounts)
        if risk_signal.entry or risk_signal.exit:
            self.signal_count += 1

        execution_started = time.perf_counter()
        await self.executor.next(snapshot, risk_signal, self.accounts)
        self.execution_ms += int((time.perf_counter() - execution_started) * 1000)
        self.last_bar = bar
        self.chart_rows.append({
            "ts_ms": bar.ts_ms,
            "open": bar.open,
            "high": bar.high,
            "low": bar.low,
            "close": bar.close,
            "ema_fast": self.signaler.values(bar.ts_ms)["ema_fast"],
            "ema_slow": self.signaler.values(bar.ts_ms)["ema_slow"],
            "signal": "entry" if risk_signal.entry else "exit" if risk_signal.exit else "",
        })

    def results(self) -> dict:
        result = self.executor.result()
        return {
            "result": result.__dict__,
            "events": self.executor.events,
            "equity": self.executor.equity,
            "chart_rows": self.chart_rows,
            "bars": len(self.chart_rows),
            "warmup_bars": self.signaler.results()["warmup_bars"],
            "signals": self.signal_count,
            "accounts": {role: {"name": account.name, "recon_count": account.recon_count} for role, account in self.accounts.items()},
            "timing": {
                "elapsed_ms": self.elapsed_ms + self.loop_ms,
                "load_ms": self.data.results()["load_ms"],
                "signal_load_ms": self.signaler.results()["signal_load_ms"],
                "core_loop_ms": self.loop_ms,
                "signal_ms": self.signal_ms,
                "execution_ms": self.execution_ms,
            },
        }


In [10]:
# Run SweepRun
# Run every sweeprun in this sweep and keep all outputs in memory.

sweep_outputs = []
for row in sweeprun_records:
    sweeprun_config = SweeprunConfig(row)
    bot_config = BotConfig(row)
    data = Data(sweeprun_config, bot_config)
    signaler = EmaCross(bot_config)
    risk = Risk(bot_config)
    executor = Tradebot(bot_config)
    runtime = SweepRuntime(sweeprun_config, bot_config, data, signaler, risk, executor)

    await runtime.init()
    await runtime.run()
    output = runtime.results()
    output["sweeprun_id"] = row.sweeprun_id
    output["sweeprun_index"] = row.sweeprun_index
    sweep_outputs.append(output)

assert sweep_outputs
assert all(output["bars"] > 0 for output in sweep_outputs)
assert all(output["result"]["cycles"] <= BotConfig(row).config.executor.max_cycles or BotConfig(row).config.executor.max_cycles == 0 for output, row in zip(sweep_outputs, sweeprun_records, strict=True))

sweep_output = sweep_outputs[0]
print(f"sweep_id={sweep_record.sweep_id} sweepruns={len(sweep_outputs)} bars={sweep_output['bars']} first_cycles={sweep_output['result']['cycles']}")


sweep_id=22 sweepruns=1 bars=525600 first_cycles=12072


In [ ]:
# Save Results
# Persist sweeprun timing and summary back to the sweeps database.

nuubot = Nuubot().setup()
try:
    by_id = {output["sweeprun_id"]: output for output in sweep_outputs}
    with nuubot.datastore.session(nuubot.config.databases.sweeps) as session:
        for row in sweeprun_records:
            output = by_id[row.sweeprun_id]
            loaded = session.get(SweeprunRow, row.sweeprun_id)
            loaded.status = "complete"
            loaded.results_json = json.dumps({"result": output["result"], "timing": output["timing"]}, sort_keys=True)
        sweep = session.get(SweepRow, sweep_record.sweep_id)
        sweep.status = "complete"
        sweep.results_json = json.dumps({
            "sweepruns": len(sweep_outputs),
            "first_result": sweep_outputs[0]["result"],
            "timing": {"elapsed_ms": sum(output["timing"]["elapsed_ms"] for output in sweep_outputs)},
        }, sort_keys=True)
        session.commit()
finally:
    nuubot.stop()

print("results saved")

In [12]:
# Raw Results
# JSON/debug output for investigation without dumping every trade event.

def iso(ts_ms: int) -> str:
    return datetime.fromtimestamp(ts_ms / 1000, tz=timezone.utc).isoformat()


def row_for(output: dict) -> dict:
    result = output["result"]
    first_ts = output["chart_rows"][0]["ts_ms"] if output["chart_rows"] else 0
    last_ts = output["chart_rows"][-1]["ts_ms"] if output["chart_rows"] else 0
    return {
        "sweeprun_id": output["sweeprun_id"],
        "sweeprun_index": output["sweeprun_index"],
        "start": iso(first_ts),
        "end": iso(last_ts),
        "duration_days": round((last_ts - first_ts) / 86_400_000, 2) if first_ts and last_ts else 0,
        "bots": 1,
        "symbols": 1,
        "bars": output["bars"],
        "warmup_bars": output["warmup_bars"],
        "signals": output["signals"],
        "pnl_pct": round(result["pnl_pct"], 6),
        "fees_pct": round(result["fees_pct"], 6),
        "trades": result["trades"],
        "wins": result["wins"],
        "losses": result["losses"],
        "win_rate_pct": round(result["wins"] / result["cycles"] * 100, 2) if result["cycles"] else 0,
        "run_ms": output["timing"]["core_loop_ms"],
        "signal_load_ms": output["timing"]["signal_load_ms"],
        "signal_ms": output["timing"]["signal_ms"],
        "load_ms": output["timing"]["load_ms"],
        "total_ms": output["timing"]["elapsed_ms"],
    }


def result_rows() -> list[dict]:
    return [row_for(output) for output in sweep_outputs]


rows = result_rows()
print(json.dumps({
    "rows": rows,
    "event_count": len(sweep_output["events"]),
    "events_preview": sweep_output["events"][:20],
    "result": sweep_output["result"],
    "accounts": sweep_output["accounts"],
}, indent=2))


{
  "rows": [
    {
      "sweeprun_id": 22,
      "sweeprun_index": 0,
      "start": "2025-01-01T00:00:00+00:00",
      "end": "2025-12-31T23:59:00+00:00",
      "duration_days": 365.0,
      "bots": 1,
      "symbols": 1,
      "bars": 525600,
      "warmup_bars": 44640,
      "signals": 24145,
      "pnl_pct": -25.229725,
      "fees_pct": 0.0,
      "trades": 12072,
      "wins": 3577,
      "losses": 8495,
      "win_rate_pct": 29.63,
      "run_ms": 950,
      "signal_load_ms": 635,
      "signal_ms": 0,
      "load_ms": 1717,
      "total_ms": 3312
    }
  ],
  "event_count": 24144,
  "events_preview": [
    {
      "event": "entry",
      "ts_ms": 1735690500000,
      "price": 93702.27,
      "reason": "ema_cross_up"
    },
    {
      "event": "exit",
      "ts_ms": 1735693560000,
      "price": 93935.46,
      "reason": "ema_cross_down",
      "pnl_pct": 0.24886270097832455,
      "fees_pct": 0.0
    },
    {
      "event": "entry",
      "ts_ms": 1735693980000,
      "price

In [13]:
# Report
# Final scroll-to-bottom chart and concise result panel.


chart_rows = sweep_output["chart_rows"]
if not chart_rows:
    print("no chart rows")
else:
    rows = result_rows()
    report = rows[0]
    display_rows = chart_rows[-5000:]
    categories = [iso(row["ts_ms"])[:16].replace("T", " ") for row in display_rows]
    candles = [[row["open"], row["close"], row["low"], row["high"]] for row in display_rows]
    ema_fast = [row["ema_fast"] for row in display_rows]
    ema_slow = [row["ema_slow"] for row in display_rows]
    marks = [
        {"coord": [i, row["close"]], "value": row["signal"], "itemStyle": {"color": "#facc15" if row["signal"] == "entry" else "#38bdf8"}}
        for i, row in enumerate(display_rows)
        if row["signal"]
    ]

    def field(label: str, value) -> str:
        return f"<div class='field'><span>{label}</span><b>{value}</b></div>"

    html = f"""
    <div id='emacross-report'>
      <div id='emacross-chart' style='height:520px;width:100%;'></div>
      <div class='summary'>
        <section>
          <h3>Runtime Stats</h3>
          {field('Period Start', report['start'])}
          {field('Period End', report['end'])}
          {field('Duration', str(report['duration_days']) + ' days')}
          {field('Symbols', report['symbols'])}
          {field('Bars', report['bars'])}
          {field('Warmup Bars', report['warmup_bars'])}
          {field('Chart Points', len(display_rows))}
        </section>
        <section>
          <h3>Performance</h3>
          {field('# Bots', report['bots'])}
          {field('Trades', report['trades'])}
          {field('Win Rate', str(report['win_rate_pct']) + '%')}
          {field('PnL', str(report['pnl_pct']) + '%')}
          {field('Fees', str(report['fees_pct']) + '%')}
        </section>
        <section>
          <h3>SweepRun Stats</h3>
          {field('Time Taken', str(report['run_ms']) + ' ms')}
          {field('Signal Load', str(report['signal_load_ms']) + ' ms')}
        </section>
      </div>
    </div>
    <style>
      #emacross-report {{ font-family: system-ui, Segoe UI, sans-serif; color: #e5e7eb; background: #111827; padding: 14px; }}
      #emacross-report .summary {{ display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 12px; margin-top: 12px; }}
      #emacross-report section {{ border: 1px solid #374151; padding: 12px; background: #0f172a; }}
      #emacross-report h3 {{ margin: 0 0 10px; font-size: 15px; }}
      #emacross-report .field {{ display: flex; justify-content: space-between; gap: 12px; padding: 5px 0; border-top: 1px solid #1f2937; font-size: 13px; }}
      #emacross-report .field span {{ color: #9ca3af; }}
      #emacross-report .field b {{ color: #f9fafb; font-weight: 600; text-align: right; }}
    </style>
    <script src='https://cdn.jsdelivr.net/npm/echarts@5/dist/echarts.min.js'></script>
    <script>
      (() => {{
        const chart = echarts.init(document.getElementById('emacross-chart'));
        chart.setOption({{
          animation: false,
          backgroundColor: '#111827',
          tooltip: {{ trigger: 'axis' }},
          legend: {{ data: ['Candles', 'EMA Fast', 'EMA Slow'], textStyle: {{ color: '#e5e7eb' }} }},
          grid: {{ left: 60, right: 30, top: 50, bottom: 80 }},
          xAxis: {{ type: 'category', data: {json.dumps(categories)}, axisLabel: {{ color: '#9ca3af' }} }},
          yAxis: {{ scale: true, axisLabel: {{ color: '#9ca3af' }}, splitLine: {{ lineStyle: {{ color: '#1f2937' }} }} }},
          dataZoom: [{{ type: 'inside' }}, {{ type: 'slider', bottom: 20 }}],
          series: [
            {{ name: 'Candles', type: 'candlestick', data: {json.dumps(candles)}, itemStyle: {{ color: '#22c55e', color0: '#ef4444', borderColor: '#22c55e', borderColor0: '#ef4444' }}, markPoint: {{ data: {json.dumps(marks)} }} }},
            {{ name: 'EMA Fast', type: 'line', data: {json.dumps(ema_fast)}, smooth: false, showSymbol: false, lineStyle: {{ color: '#facc15', width: 1.5 }} }},
            {{ name: 'EMA Slow', type: 'line', data: {json.dumps(ema_slow)}, smooth: false, showSymbol: false, lineStyle: {{ color: '#38bdf8', width: 1.5 }} }}
          ]
        }});
      }})();
    </script>
    """
    report_path = Path("workspace/results/emacross-report.html")
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(html, encoding="utf-8")
    display(IFrame(str(report_path), width="100%", height=900))
